In [2]:
import accelerate
print(accelerate.__version__)


1.12.0


In [1]:
!pip install --upgrade accelerate transformers


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
import torch

# Model setup
model_path = "ibm-granite/granite-4.0-tiny-preview"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto" if device=="cuda" else None,
    torch_dtype=torch.bfloat16 if device=="cuda" else torch.float32
)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# 5 people profiles (add more if needed)
people = [
    {
        "full_name": "Mohamed Abdelaaty",
        "email": "mohamedabdelaaty@digitalfortresseg.com",
        "job_title": "Managing Director, SOC",
        "responsibilities": "Managing Digitalfortress",
        "experience": "1–3 years",
        "interests": ["Cybersecurity & online safety", "Travel & outdoor activities", "Reading/books", "Finance & investing", "Health & fitness"],
        "communication_style": "straight to the point(concise, bullet points)",
        "age" : ["25-34"],
        "location": "United Arab Emirates"
    },
    {
        "full_name": "Omar Nader Shams",
        "email": "omers9721@gmail.com",
        "job_title": "GRC Specialist",
        "responsibilities": "Compliance",
        "experience": "Less than 1 year",
        "interests": ["Cybersecurity & online safety", "Technology & gadgets", "Travel & outdoor activities", "Sports", "Reading/books", "Gaming", "Health & fitness"],
        "communication_style": "formal and professional",
        "age" : ["25-34"],
        "location": "Egypt"
    },
    {
        "full_name": "Mostafa Abdelftha Soliman Mahna",
        "email": "mostafamahna1@gmail.com",
        "job_title": "SOC Tier1",
        "responsibilities": "Public relations",
        "experience": "1–3 years",
        "interests": ["Cybersecurity & online safety", "Technology & gadgets, Sports", "Finance & investing", "Health & fitness"],
        "communication_style": "friendly and conversational",
        "age" : ["Under 25"],
        "location": "Cairo, Egypt"
    }
    
]

# Build the tailored prompt
person_prompts = []
for p in people:
    person_prompts.append(f"""
Profile:
- Name: {p['full_name']}
- Email: {p['email']}
- Job Title: {p['job_title']}
- Main responsibilities: {p['responsibilities']}
- Experience: {p['experience']}
- Interests: {', '.join(p['interests'])}
- Preferred communication style: {p['communication_style']}
- Age: {p['age']}
- Location: {p['location']}
""")

person_prompt_str = "\n---\n".join(person_prompts)

full_prompt = f"""
You are a cybersecurity simulation assistant. Your task is to generate simulated phishing emails for **defensive training only**.

You will receive the profile of 3 individuals below. For each:
- Generate **1 simulated phishing email** tailored to their role, interests, and communication style.
- Each output must include:
  * Scenario context
  * Simulated sender (DigitalFortresseg.com) 
  * Subject
  * Body (concise and mindful of communication style)
  * A placeholder fake link labeled [SIMULATED PHISHING LINK]
  * Phishing technique type
  * Must be clearly labeled: SIMULATED PHISHING EXERCISE FOR DEFENSE TRAINING

Profiles:
{person_prompt_str}

Produce one example per profile in structured format.
"""

conv = [{"role": "user", "content": full_prompt}]

# Encode for model
inputs = tokenizer.apply_chat_template(
    conv,
    return_tensors="pt",
    thinking=True,
    return_dict=True,
    add_generation_prompt=True
).to(device)

# Generate
set_seed(42)
output = model.generate(
    **inputs,
    max_new_tokens=8192
)

response = tokenizer.decode(
    output[0, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)


`torch_dtype` is deprecated! Use `dtype` instead!
The fast path is not available because one of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

<think>To create simulated phishing emails for defensive training, I will craft emails tailored to each individual's role, interests, and communication style. The emails will include a scenario context, sender details, subject, body, a fake link, and a label indicating the training exercise.

### Profile 1: Mohamed Abdelaaty

**Scenario Context:**
Mohamed, as the Managing Director of DigitalFortress, you've been invited to a conference in Dubai to discuss the latest cybersecurity trends. The sender claims to be from the conference organizers, offering exclusive insights and networking opportunities.

**Sender:**
DigitalFortresseg.com

**Subject:**
Exclusive Invitation: DigitalFortress Conference Insights

**Body:**
Dear Mohamed,

We hope this message finds you well. As the Managing Director of DigitalFortress, your insights are highly valued in the cybersecurity community.

We are excited to invite you to the upcoming DigitalFortress Conference in Dubai, where you'll have the opportuni

In [10]:
!pip uninstall --yes numpy







Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4


In [11]:
!pip install numpy==1.26.4


  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [1]:
import numpy
print(numpy.__version__)  # should print 1.26.4


1.26.4


In [2]:
from vllm import LLM, SamplingParams

# Initialize the model
llm = LLM(model="microsoft/Phi-4-mini-instruct", trust_remote_code=True)

# 5 people profiles
people = [
    {
        "full_name": "Mohamed Abdelaaty",
        "email": "mohamedabdelaaty@digitalfortresseg.com",
        "job_title": "Managing Director, SOC",
        "responsibilities": "Managing Digitalfortress",
        "experience": "1–3 years",
        "interests": ["Cybersecurity & online safety", "Travel & outdoor activities", "Reading/books", "Finance & investing", "Health & fitness"],
        "communication_style": "straight to the point(concise, bullet points)",
        "age" : ["25-34"],
        "location": "United Arab Emirates"
    },
    {
        "full_name": "Omar Nader Shams",
        "email": "omers9721@gmail.com",
        "job_title": "GRC Specialist",
        "responsibilities": "Compliance",
        "experience": "Less than 1 year",
        "interests": ["Cybersecurity & online safety", "Technology & gadgets", "Travel & outdoor activities", "Sports", "Reading/books", "Gaming", "Health & fitness"],
        "communication_style": "formal and professional",
        "age" : ["25-34"],
        "location": "Egypt"
    },
    {
        "full_name": "Mostafa Abdelftha Soliman Mahna",
        "email": "mostafamahna1@gmail.com",
        "job_title": "SOC Tier1",
        "responsibilities": "Public relations",
        "experience": "1–3 years",
        "interests": ["Cybersecurity & online safety", "Technology & gadgets, Sports", "Finance & investing", "Health & fitness"],
        "communication_style": "friendly and conversational",
        "age" : ["Under 25"],
        "location": "Cairo, Egypt"
    }
    
]

# Build the combined prompt text
profile_prompts = []
for p in people:
    profile_prompts.append(f"""
Profile:
- Name: {p['full_name']}
- Email: {p['email']}
- Job Title: {p['job_title']}
- Responsibilities: {p['responsibilities']}
- Experience: {p['experience']}
- Interests: {', '.join(p['interests'])}
- Communication Style: {p['communication_style']}
- Age: {p['age']}
- Location: {p['location']}
""")

profiles_text = "\n---\n".join(profile_prompts)

# Final instruction prompt
full_prompt = f"""
You are a cybersecurity simulation assistant. Your task is to generate simulated phishing emails for **defensive training only**.

You will receive the profile of 3 individuals below. For each:
- Generate **1 simulated phishing email** tailored to their role, interests, and communication style.
- Each output must include:
  * Scenario context
  * Simulated sender (DigitalFortresseg.com) 
  * Subject
  * Body (concise and mindful of communication style)
  * A placeholder fake link labeled [SIMULATED PHISHING LINK]
  * Phishing technique type
  * Must be clearly labeled: SIMULATED PHISHING EXERCISE FOR DEFENSE TRAINING

Profiles:
{profiles_text}

Produce one example per profile in structured format.
"""

messages = [
    {"role": "user", "content": full_prompt}
]

# Sampling settings
sampling_params = SamplingParams(
    max_tokens=500,
    temperature=0.0
)

# Run the model
output = llm.chat(messages=messages, sampling_params=sampling_params)

# Print the model's text
print(output[0].outputs[0].text)


INFO 12-31 16:25:48 [utils.py:253] non-default args: {'trust_remote_code': True, 'disable_log_stats': True, 'model': 'microsoft/Phi-4-mini-instruct'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


config.json: 0.00B [00:00, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-4-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 12.33it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 14.42it/s]


(EngineCore_DP0 pid=94929) INFO 12-31 16:27:37 [gpu_model_runner.py:4587] Graph capturing finished in 8 secs, took 0.52 GiB
(EngineCore_DP0 pid=94929) INFO 12-31 16:27:37 [core.py:259] init engine (profile, create kv cache, warmup model) took 38.85 seconds
INFO 12-31 16:27:38 [llm.py:360] Supported tasks: ['generate']
INFO 12-31 16:27:40 [chat_utils.py:590] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

**SIMULATED PHISHING EXERCISE FOR DEFENSE TRAINING**

---

**Profile: Mohamed Abdelaaty**

*Scenario context:* Mohamed receives an email that appears to be from Digitalfortress, asking for his immediate attention regarding a new cybersecurity initiative.

*Simulated sender:* DigitalFortresseg.com

*Subject:* Urgent: New Cybersecurity Initiative - Immediate Action Required

*Body:*
- Subject: New Cybersecurity Initiative
- Body: 
  - We are launching a new cybersecurity initiative to enhance our online safety.
  - Please review the attached document and provide your feedback by EOD.
  - [SIMULATED PHISHING LINK]
  - Action Required: Click here to access the document.

*Phishing technique type:* Urgency and authority

---

**Profile: Omar Nader Shams**

*Scenario context:* Omar receives an email that appears to be from a compliance department, asking for his immediate attention regarding a new compliance policy.

*Simulated sender:* DigitalFortresseg.com

*Subject:* Compliance Update: Im